# 02 -- Analisis Exploratorio (EDA)

**Objetivo**: Explorar distribuciones de tiempos de resolucion, volumen por agencia y municipio, patrones temporales y calidad de datos en las solicitudes NYC 311 de 2024.

**Entrada**: `data/processed/nyc311_clean.parquet`

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_parquet(Path('../data/processed/nyc311_clean.parquet'))
print(f'Datos cargados: {len(df):,} filas x {len(df.columns)} columnas')

## 1. Distribucion de Tiempos de Resolucion

El tiempo de resolucion es la metrica central de eficiencia operacional. Analizamos su distribucion para identificar la forma, outliers y patrones.

In [ ]:
# Calcular tiempos de resolucion para solicitudes cerradas
df['resolution_hours'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600
df['resolution_days'] = df['resolution_hours'] / 24

cerradas = df[df['resolution_days'].notna() & (df['resolution_days'] > 0)]
print(f'Solicitudes con tiempo de resolucion: {len(cerradas):,} ({len(cerradas)/len(df):.1%})')
print(f'\nEstadisticas de resolucion (dias):')
cerradas['resolution_days'].describe().round(2)

In [ ]:
# Histograma de tiempos de resolucion (truncado a 60 dias para visibilidad)
fig = px.histogram(
    cerradas[cerradas['resolution_days'] <= 60],
    x='resolution_days',
    nbins=60,
    title='Distribucion de Tiempos de Resolucion (truncado a 60 dias)',
    labels={'resolution_days': 'Dias de Resolucion'},
    color_discrete_sequence=['#3B82F6']
)
fig.add_vline(x=cerradas['resolution_days'].median(), line_dash='dash', line_color='#F59E0B',
              annotation_text=f'Mediana: {cerradas["resolution_days"].median():.1f} dias')
fig.update_layout(height=400, template='plotly_dark', showlegend=False)
fig.show()

## 2. Volumen por Agencia

Identificamos las agencias con mayor carga de trabajo. Esto nos ayuda a focalizar el analisis de eficiencia.

In [ ]:
agency_vol = df['agency_name'].value_counts().head(15).reset_index()
agency_vol.columns = ['agencia', 'solicitudes']

fig = px.bar(
    agency_vol,
    x='solicitudes', y='agencia',
    orientation='h',
    title='Top 15 Agencias por Volumen de Solicitudes',
    labels={'solicitudes': 'Solicitudes', 'agencia': ''},
    color='solicitudes',
    color_continuous_scale='Blues'
)
fig.update_layout(height=500, template='plotly_dark', yaxis={'categoryorder': 'total ascending'})
fig.show()

## 3. Distribucion Geografica por Municipio

In [ ]:
borough_vol = df['borough'].value_counts().reset_index()
borough_vol.columns = ['municipio', 'solicitudes']

fig = px.bar(
    borough_vol,
    x='municipio', y='solicitudes',
    title='Solicitudes por Municipio (Borough)',
    labels={'solicitudes': 'Solicitudes', 'municipio': 'Municipio'},
    color='municipio',
    color_discrete_sequence=['#3B82F6', '#8B5CF6', '#10B981', '#F59E0B', '#6B7280']
)
fig.update_layout(height=400, template='plotly_dark', showlegend=False)
fig.show()

## 4. Tipos de Queja mas Frecuentes

In [ ]:
complaint_vol = df['complaint_type'].value_counts().head(20).reset_index()
complaint_vol.columns = ['tipo_queja', 'solicitudes']
complaint_vol['pct'] = (complaint_vol['solicitudes'] / len(df) * 100).round(1)
complaint_vol['pct_acumulado'] = complaint_vol['pct'].cumsum()

fig = px.bar(
    complaint_vol,
    x='solicitudes', y='tipo_queja',
    orientation='h',
    title='Top 20 Tipos de Queja -- Analisis Pareto',
    labels={'solicitudes': 'Solicitudes', 'tipo_queja': ''},
    color='solicitudes',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=600, template='plotly_dark', yaxis={'categoryorder': 'total ascending'})
fig.show()

print(f'\nLos top 20 tipos de queja representan el {complaint_vol["pct"].sum():.1f}% del total')

## 5. Patrones Temporales

In [ ]:
# Serie temporal mensual
df['created_month'] = df['created_date'].dt.to_period('M').astype(str)
monthly = df.groupby('created_month').size().reset_index(name='solicitudes')

fig = px.line(
    monthly,
    x='created_month', y='solicitudes',
    title='Volumen Mensual de Solicitudes 311 -- 2024',
    labels={'created_month': 'Mes', 'solicitudes': 'Solicitudes'},
    markers=True
)
fig.update_traces(line_color='#3B82F6', marker_color='#06B6D4')
fig.update_layout(height=400, template='plotly_dark')
fig.show()

In [ ]:
# Heatmap dia de semana x hora
DOW_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
DOW_ES = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']

df['dow'] = pd.Categorical(df['created_date'].dt.day_name(), categories=DOW_ORDER, ordered=True)
df['hour'] = df['created_date'].dt.hour

heatmap_data = df.groupby(['dow', 'hour']).size().unstack(fill_value=0)
heatmap_data.index = DOW_ES

fig = px.imshow(
    heatmap_data,
    title='Solicitudes por Dia de Semana y Hora',
    labels=dict(x='Hora', y='Dia', color='Solicitudes'),
    color_continuous_scale='Blues',
    aspect='auto'
)
fig.update_layout(height=400, template='plotly_dark')
fig.show()

## 6. Canales de Recepcion

In [ ]:
channel_vol = df['open_data_channel_type'].value_counts().reset_index()
channel_vol.columns = ['canal', 'solicitudes']
channel_vol['pct'] = (channel_vol['solicitudes'] / len(df) * 100).round(1)

fig = px.bar(
    channel_vol,
    x='canal', y='solicitudes',
    title='Solicitudes por Canal de Recepcion',
    labels={'solicitudes': 'Solicitudes', 'canal': 'Canal'},
    color='canal',
    color_discrete_sequence=['#3B82F6', '#8B5CF6', '#10B981', '#F59E0B', '#EF4444']
)
fig.update_layout(height=400, template='plotly_dark', showlegend=False)
fig.show()

channel_vol

## 7. Hallazgos Clave del EDA

1. **Distribucion sesgada**: Los tiempos de resolucion tienen una cola derecha larga -- la mediana es significativamente menor que la media
2. **Concentracion de agencias**: Un punado de agencias maneja la mayoria de las solicitudes
3. **Brooklyn y Queens** lideran en volumen, reflejando densidad poblacional
4. **Patron temporal claro**: Mayor actividad en horas laborales (9-17h), dias laborales, y meses de verano
5. **Canal digital creciente**: Telefono sigue dominando pero los canales digitales representan una porcion significativa

---

**Siguiente**: [03_analisis_sla.ipynb](03_analisis_sla.ipynb) -- Analisis de cumplimiento SLA por agencia y factores de prediccion.